In [ ]:
# TO DO: Ir guardando los resultados optimos cada x iteraciones
# agregar tema de ordenes fijas

# Parámetros

In [ ]:
max_iters = 10000

# Data x hora

In [ ]:
# ALg semiautomático

# AProvecha lo construido para operar de forma semiautomática


#1. Buscar resistencias y soportes (se puede actualizar por ej cada una semana)....esas serán las ordenes de compra durante toda la semana
# 2. Conectarse con mt5 y ejecutar y hacer el trallingstop

In [ ]:
# Ejecutar con revenAI
# version yf = 0.2.65

# V2: Cambios el 260111...rediseño de macroalgorítmo y continuación de desarrollo

In [ ]:
print('Conexion con mt5 en mt5.ipynb [en este mismo proyecto]')
print('Llevado a la práctica en 00_Conexion_mt5.ipynb')

In [ ]:
#sys.exit("Entre 'df_oc_all base inicial' y 'df_oc_all base final', hay status que pasan de Abierta a Activa...eso no puede ocurrir [ver id 11]")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import itertools
import tqdm
import random
import warnings
import sys
import pickle

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

import mplfinance as mpf
from typing import Union

In [ ]:
opcion = 2 # 0 solo actualiza valores y 1 solo ejecuta el algorítmo..2: Realiza ambas cosas

In [ ]:
minimo_lotaje = True
continuar_por_ahora = False # Hasta resolver temas de factibilidad como el traspaso de Status Abierta a Activa

In [ ]:
conexion_mt5 = False # No hay conexión MT5 por ahora

In [ ]:
if opcion == 0:
    actualizar_valores = True
    ejecutar_proceso = False
elif opcion == 1:
    actualizar_valores = False
    ejecutar_proceso = True
elif opcion == 2:
    actualizar_valores = True
    ejecutar_proceso = True

In [ ]:
carpeta_data = "../Data/"

In [ ]:
import MetaTrader5 as mt5
import pandas as pd


# Inicializar MT5 (Inicialmente, debe conextarse para obtener datos de precio)
mt5.initialize()

# Initialize MT5 connection
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

In [ ]:

valores = ['BTCUSD', 'ETHUSD', 'TSLA', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'NFLX']
for valor in valores:
    if not actualizar_valores:
        break
    
    # Nombre del símbolo (depende de tu broker, a veces es BTCUSD, BTCUSD.m, etc.)
    symbol = valor
    print('SY', symbol)

    # Seleccionar símbolo
    mt5.symbol_select(symbol, True)

    # Descargar datos: 100 velas de 1 hora (TIMEFRAME_H1)
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 1000) # Max 60000 (en 1000, hay 41 días)
    
    # Prueba extraccion x minuto (max 99999): 69.44 días
    #rates_min = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M1, 0, 99999) # Para obtener el último bloque horario "abierto"
    #df_min = pd.DataFrame(rates_min)
    
    #display(df_min)
    #sys.exit()
    
    # Convertir a DataFrame
    df = pd.DataFrame(rates)
    try:
        df['time'] = pd.to_datetime(df['time'], unit='s') # A veces falla
    except Exception as e:
        print(f'Error al convertir tiempo para {valor}: {e}')
        continue
    df.columns = ['DateTime', 'Open', 'High', 'Low', 'Close', 'Tick_Volume', 'Spread', 'Real_Volume'] # Campos selecicionados
    
    display(f'df {len(df)}', df.tail())

    data_old = pd.DataFrame() # Append
    if f'{valor}.csv' in os.listdir(carpeta_data):
        data_old = pd.read_csv(f'{carpeta_data}{valor}.csv')
        data_old['DateTime'] = pd.to_datetime(data_old['DateTime'])
        df['DateTime'] = pd.to_datetime(df['DateTime'])
    data = pd.concat([df, data_old]).drop_duplicates(subset=['DateTime']).reset_index(drop=True) # Los nuevos tienen prioridad, ya que la última hora mostrada en df, está "abierta"...eso hará que, ese bloque horario pase a data_old y después sea pisada por un nuevo df
    data.to_csv(f'{carpeta_data}{valor}.csv', index=False)


# Funciones

In [ ]:
if not ejecutar_proceso:
    sys.exit()

In [ ]:
def pickle_act(file_name, variable = None, mode = 'open', eliminar_si_problemas = False): #260120
    
    """
    Guarda o carga una variable utilizando la biblioteca pickle.

    Parameters:
        - file_path (str): La ruta al archivo pickle.
        - variable: La variable a guardar (si mode='save') o None (si mode='open').
        - mode (str): 'save' para guardar la variable, 'open' para cargar la variable.

    Returns:
        La variable cargada si mode='open' o None si mode='save'.
    """
    
    dic_mode = {'save': 'wb', 'open': 'rb'}
    #print('file_name', file_name)

    while True:
        try:
            with open(f'{file_name}.pkl', dic_mode[mode]) as file:
                if mode == 'save':
                    pickle.dump(variable, file)
                    return None
                else:
                    #print('file en funciones transversales', f'{file_name}.pkl')
                    if eliminar_si_problemas:
                        try:
                            variable = pickle.load(file)
                        except:
                            os.remove(f'{file_name}.pkl')
                            return pd.DataFrame()
                    else:
                        try:
                            variable = pickle.load(file)
                        except Exception as e:
                            print('Problema en pickle_act 1:', e)
                            try:
                                with open(f'{file_name}.pkl', 'wb') as f:
                                    pickle.dump(df, f)
                            except Exception as e:
                                print('Problema en pickle_act 2:', e)
                                print(f"Intentar ejecutar with open(f'{file_name}.pkl', 'wb') as f: pickle.dump(df, f)")
                                sys.exit('A')
                    return variable
        except Exception as e:
            print(f'Problema en pickle_act 3: {e}')
            sys.exit()
            #time.sleep(10)
    
    return None

In [ ]:
# Comentada con typeHints
def calcular_distancias(df: pd.DataFrame, find_low: bool = True, find_high: bool = True) -> pd.DataFrame:
    '''
    Recibe un dattaframe con precios, incluido una columna "t" que va de 0 a 1 (normalizada). Se incluyen además el High y el Low Price en cada fila
    Además, recibe dos booleanos find_low y find_high, que indican si se deben calcular las distancias para Low y High respectivamente. Ambas alternativas son idénticas
    Devuelve el dataframe con las columnas agregadas:
    - Si find_low es True:
        - Low_left: distancia al soporte más cercano a la izquierda (en t) que contenga el Low actual
        - Low_right: distancia al soporte más cercano a la derecha (en t) que contenga el Low actual
    - Si find_high es True:
        - High_left: distancia a la resistencia más cercana a la izquierda (en t) que contenga el High actual
        - High_right: distancia a la resistencia más cercana a la derecha (en t) que contenga el High actual
    Las distancias se miden en unidades de t (tiempo normalizado)
    '''
    n = len(df)
    df = df.copy()
    
    # Activación de campos de outputs (inicialmente como nans)
    if find_low:
        df["Low_left"] = np.nan
        df["Low_right"] = np.nan
    if find_high:
        df["High_left"] = np.nan
        df["High_right"] = np.nan

    max_t = df["t"].max()

    for i in tqdm.tqdm(range(n)): # Para cada registro
        low_val = df.loc[i, "Low"] # Se miden precios Low y High
        high_val = df.loc[i, "High"]
        t_i = df.loc[i, "t"] # Tiempo normalizado actual
        
        if find_low: # Si find_low es True
            # Buscar hacia la izquierda (Low)
            for j in range(i-1, -1, -1): # Se busca el registro a la izquierda (de t hacia atrás) donde low_val esté entre Low y High
                if df.loc[j, "Low"] <= low_val <= df.loc[j, "High"]:
                    df.loc[i, "Low_left"] = t_i - df.loc[j, "t"] # Distancia en t y break (es el punto más cercano)
                    break

            # Buscar hacia la derecha (Low)
            for j in range(i+1, n): # Análogo para la derecha, de t hacia adelante
                if df.loc[j, "Low"] <= low_val <= df.loc[j, "High"]:
                    df.loc[i, "Low_right"] = df.loc[j, "t"] - t_i
                    break

        if find_high: # Análogo para High
            # Buscar hacia la izquierda (High)
            for j in range(i-1, -1, -1):
                if df.loc[j, "Low"] <= high_val <= df.loc[j, "High"]:
                    df.loc[i, "High_left"] = t_i - df.loc[j, "t"]
                    break

            # Buscar hacia la derecha (High)
            for j in range(i+1, n):
                if df.loc[j, "Low"] <= high_val <= df.loc[j, "High"]:
                    df.loc[i, "High_right"] = df.loc[j, "t"] - t_i
                    break

    # Rellenar los NaN con t o max(t) - t (casos en los que se llega al final, a la derecha o a la izquierda sin encontrar un registro que contenga el precio)
    if find_low:
        df["Low_left"] = df["Low_left"].fillna(df["t"])
        df["Low_right"] = df["Low_right"].fillna(max_t - df["t"])
    if find_high:
        df["High_left"] = df["High_left"].fillna(df["t"])
        df["High_right"] = df["High_right"].fillna(max_t - df["t"])

    return df


In [ ]:
def obtener_df_extremos(df_0: pd.DataFrame, k: float, n: float, N: int, M: int, OCP: int, conjunto_N = set(), prints_general = False):
    '''
    Recibe el df con los datos de velas (OHLCV) y calcula los soportes óptimos
    # Recibe parámetros para otorgar puntaje a esos soportes óptimos
    '''
    
    # 1. Cálculo de y: Distancia a mínimos anteriores y futuros
    df_extremos = calcular_distancias(df_0) # Calcula las distancias a los mínimos anteriores y futuros
    # Ponderadores (audios wsp personal 251002)
    if 'High_left' in df_extremos.columns: # Depende de si se ocupa find_high, también se pueden considerar las resistencias
        df_extremos['y'] = df_extremos['High_left'] + df_extremos['Low_left'] + k * (df_extremos['High_right'] + df_extremos['Low_right'])
    else:
        df_extremos['y'] = df_extremos['Low_left'] + k * df_extremos['Low_right']
    
    # 2. Cálculo de w: Ponderación por tiempo (se ponderan más los recientes)
    df_extremos['w'] = df_extremos['t'] ** n # t min es 0 y t max es 1 (ya están normalizados, por lo que no hay problema) > t ** n = 0 y t ** n = 1
    
    # BUSQUEDA DE SOPORTES OPTIMOS (Potenciales. En el nuevo lenguaje, sería buscar...)
    # El objetivo es, con M puntos potenciales, llegar a que OCE + OCP = N soportes óptimos [OCE: Ordenes de compra en espera]

    if prints_general:
        print('N-M', N, M)

    p_min, p_max = df_extremos['Low'].min(), df_extremos['Low'].max()
    if prints_general:
        print('p_min y p_max', p_min, p_max)
    
    # Se debe entregar conjunto_N y sizes
    #conjunto_M = set(np.linspace(p_min, p_max, M).tolist()) # M precios equidistantes entre p_min y p_max (potenciales soportes)
    
    # M valores que distribuyen Uniforme entre p_min y p_max
    print('Prueba conjuntoM distribuye Uniforme') # 260119
    conjunto_M = set(np.random.uniform(p_min, p_max, M).tolist())
    ordenes_en_espera = N - OCP # Cantidad de ordenes en espera que deben estar en conjunto_N (N [total] - OCP [ordenes de compra planificadas (ya activadas en la plataforma)])
    L_conjunto_N = len(conjunto_N)
    if L_conjunto_N <= ordenes_en_espera:
        # Rellenar conjunto_N con nuevos soportes aleatorios desde conjunto_M
        delta = ordenes_en_espera - L_conjunto_N
        print('delta', delta, ordenes_en_espera, L_conjunto_N)
        conjunto_M = set(np.random.uniform(p_min, p_max, M).tolist())
        nuevos_soportes = set(random.sample(list(conjunto_M), delta))
        conjunto_N = conjunto_N.union(nuevos_soportes)
    else:
        # Quitar elementos random de conjunto_N para que su tamaño sea ordenes_en_espera
        # No debería entrar aqui
        # En caso de algorítmo semiautomático
        delta = L_conjunto_N - ordenes_en_espera
        # A conjunto N hay que removerle delta elementos aleatorios
        elementos_a_remover = set(random.sample(list(conjunto_N), delta))
        conjunto_N = conjunto_N.difference(elementos_a_remover)
        #sys.exit('Validacion de no entrar 01')
        
    # Remover de conjunto_M los elementos que ya están en conjunto_N
    conjunto_M = conjunto_M.difference(conjunto_N)
    
    # Complementar M si es necesario 260120
    delta_M = M - len(conjunto_M)
    if delta_M > 0:
        nuevos_soportes_M = set(np.random.uniform(p_min, p_max, delta_M).tolist())
        conjunto_M = conjunto_M.union(nuevos_soportes_M)
    
    if (len(conjunto_N) != N) or (len(conjunto_M) != M):
        print('len conjunto_N', len(conjunto_N), 'len conjunto_M', len(conjunto_M))
        print('N y M esperados', N, M)
        sys.exit('Error en tamaños de conjunto_N y conjunto_M en obtener_df_extremos')
    # Hasta aqui bloque agregado el 260120
    
    if prints_general:
        print('L(N) * L(M)', len(conjunto_N), len(conjunto_M), len(conjunto_N) * (len(conjunto_M)))

    return df_extremos, conjunto_N, conjunto_M



In [ ]:
def obtener_df_extremos_V2(df_0: pd.DataFrame, k: float, n: float, N: int, M: int, OCP: int, conjunto_N = set(), prints_general = False):
    '''
    Recibe el df con los datos de velas (OHLCV) y calcula los soportes óptimos
    # Recibe parámetros para otorgar puntaje a esos soportes óptimos
    '''
    
    # 1. Cálculo de y: Distancia a mínimos anteriores y futuros
    df_extremos = calcular_distancias(df_0) # Calcula las distancias a los mínimos anteriores y futuros
    # Ponderadores (audios wsp personal 251002)
    if 'High_left' in df_extremos.columns: # Depende de si se ocupa find_high, también se pueden considerar las resistencias
        df_extremos['y'] = df_extremos['High_left'] + df_extremos['Low_left'] + k * (df_extremos['High_right'] + df_extremos['Low_right'])
    else:
        df_extremos['y'] = df_extremos['Low_left'] + k * df_extremos['Low_right']
    
    # 2. Cálculo de w: Ponderación por tiempo (se ponderan más los recientes)
    df_extremos['w'] = df_extremos['t'] ** n # t min es 0 y t max es 1 (ya están normalizados, por lo que no hay problema) > t ** n = 0 y t ** n = 1
    
    # BUSQUEDA DE SOPORTES OPTIMOS (Potenciales. En el nuevo lenguaje, sería buscar...)
    # El objetivo es, con M puntos potenciales, llegar a que OCE + OCP = N soportes óptimos [OCE: Ordenes de compra en espera]

    if prints_general:
        print('N-M', N, M)

    p_min, p_max = df_extremos['Low'].min(), df_extremos['Low'].max()
    if prints_general:
        print('p_min y p_max', p_min, p_max)
    
    # Se debe entregar conjunto_N y sizes
    #conjunto_M = set(np.linspace(p_min, p_max, M).tolist()) # M precios equidistantes entre p_min y p_max (potenciales soportes)
    
    # M valores que distribuyen Uniforme entre p_min y p_max
    print('Prueba conjuntoM distribuye Uniforme') # 260119
    #conjunto_M = set(np.random.uniform(p_min, p_max, M).tolist())
    ordenes_en_espera = N - OCP # Cantidad de ordenes en espera que deben estar en conjunto_N (N [total] - OCP [ordenes de compra planificadas (ya activadas en la plataforma)])
    L_conjunto_N = len(conjunto_N)
    if L_conjunto_N <= ordenes_en_espera:
        # Rellenar conjunto_N con nuevos soportes aleatorios desde conjunto_M
        delta = ordenes_en_espera - L_conjunto_N
        nuevos_soportes = set(np.random.uniform(p_min, p_max, delta).tolist())
        print('delta', delta, ordenes_en_espera, L_conjunto_N)
        #nuevos_soportes = set(random.sample(list(conjunto_M), delta))
        conjunto_N = conjunto_N.union(nuevos_soportes)
    else:
        # Quitar elementos random de conjunto_N para que su tamaño sea ordenes_en_espera
        # No debería entrar aqui
        # En caso de algorítmo semiautomático
        delta = L_conjunto_N - ordenes_en_espera
        # A conjunto N hay que removerle delta elementos aleatorios
        elementos_a_remover = set(random.sample(list(conjunto_N), delta))
        conjunto_N = conjunto_N.difference(elementos_a_remover)
        #sys.exit('Validacion de no entrar 01')
        
    # Remover de conjunto_M los elementos que ya están en conjunto_N
    #conjunto_M = conjunto_M.difference(conjunto_N)
    
    # Complementar M si es necesario 260120
    #delta_M = M - len(conjunto_M)
    #if delta_M > 0:
    #    nuevos_soportes_M = set(np.random.uniform(p_min, p_max, delta_M).tolist())
    #    conjunto_M = conjunto_M.union(nuevos_soportes_M)
    
    if (len(conjunto_N) != N):
        print('len conjunto_N', len(conjunto_N))
        print('N esperados', N)
        sys.exit('Error en tamaños de conjunto_N  en obtener_df_extremos')
    # Hasta aqui bloque agregado el 260120
    
    if prints_general:
        print('L(N)', len(conjunto_N))

    return df_extremos, conjunto_N



In [ ]:
def graficar_df_extremos(df_extremos, graficar = False):
    
    if not graficar:
        return None
    # Grafica en una matriz de 2x1, Low e y
    """
    plt.figure(figsize=(21, 10))
    #plt.subplot(1, 1, 1)
    plt.plot(df_extremos['DateTime'], df_extremos['Low'], linestyle='-', color='b', label='Low')
    plt.plot(df_extremos['DateTime'], df_extremos['High'], linestyle='-', color='g', label='High')
    plt.xlabel('DateTime')
    plt.ylabel('Low')
    plt.title('Gráfico de Low en función de DateTime')
    plt.grid()
    plt.legend()
    plt.show()
    """
    print('TO DO: Poner los parámetros asociados en cada curva, ej y = izq + lambda * der....w = t ** n...')
    plt.figure(figsize=(21, 10))
    plt.subplot(2, 2, 1)
    plt.plot(df_extremos['DateTime'], df_extremos['y'], linestyle='-', color='r')
    plt.xlabel('DateTime')
    plt.ylabel('Peso')
    plt.title('y (a mayor y, mayor distancia izq y derecha con otras velas)')
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.subplot(2, 2, 2)
    plt.plot(df_extremos['DateTime'], df_extremos['w'], linestyle='-', color='m')
    plt.xlabel('DateTime')
    plt.ylabel('Peso')
    plt.title('w (a mayor w, más reciente es la vela)')
    plt.grid()
    plt.legend()
    plt.subplot(2, 2, 3)
    plt.plot(df_extremos['DateTime'], df_extremos['h_dist'], linestyle='-', color='c')
    plt.xlabel('DateTime')
    plt.ylabel('Peso')
    plt.title('h_dist (mayor h_dist implica más cercanía con el soporte seleccionado más cercano)')
    plt.grid()
    plt.subplot(2, 2, 4)
    plt.plot(df_extremos['DateTime'], df_extremos['z'], linestyle='-', color='black')
    plt.xlabel('DateTime')
    plt.ylabel('Peso')
    plt.title('z (w * h_dist * y)')
    plt.grid()
    plt.legend()
    plt.show()

    return None

In [ ]:
# Al Low en df extremos, asignarle el soporte más cercano en conjunto_N
# Revisadas en nueva versión
def asignar_soporte(df, soportes):
    # Devuelve la distancia entre el Low y el soporte más cercano
    df = df.copy()
    df['soporte'] = df['Low'].apply(lambda x: min(soportes, key=lambda s: abs(s - x))) # Entrega s en soportes
    return df

def calcular_FO(df_extremos, conjunto_N, lambda_ponderador):

    biobjetivo = True
    
    df_extremos = asignar_soporte(df_extremos, conjunto_N) # Se asigna el soporte más cercano
    df_extremos['dist'] = (df_extremos['soporte'] - df_extremos['Low']) ** 2 # Se Penalizan distancias miuy grandes (corrección 260111: ANtes era dist_soporte - Low, pero en la función ya calcula la distancia)
    dist_max = df_extremos['dist'].max()
    df_extremos['h_dist'] = 1 - df_extremos['dist'] / dist_max # Normaliza, buscando que las distancias más pequeñas tengan mayor H_dist
    df_extremos
    
    # Calcular la función objetivo

    df_extremos['z'] = df_extremos['y'] * df_extremos['w'] * df_extremos['h_dist'] # Multiplicación de los tres términos
    # z debe ser grande
    # y: mejor representación de soporte (más aislado)
    # w: mas cercano (premia a las velas más recientes)
    # h_dist: mejor ajuste al soporte del conjunto n (más cercano al n asignado)

    L_n = list(conjunto_N)
    L_n.sort()
    H_n = [L_n[i] - L_n[i-1] for i in range(1, len(L_n))] # Distancias entre los soportes del conjunto_N

    # calcula promedio de conjunto H_n
    promedio_conjunto_H = np.mean(H_n)
    # desviación estándar de conjunto H_n
    desviacion_conjunto_H = np.std(H_n)

    # cv del conjunto H_n
    cv_conjunto_H = desviacion_conjunto_H / promedio_conjunto_H

    # cv_conjunto_N debe ser pequeño
    
    if biobjetivo:
        #print("df_extremos['z'].sum() / len(df_extremos)", df_extremos['z'].sum() / len(df_extremos), df_extremos['z'].sum(), len(df_extremos))
        #print('cv_conjunto_H', cv_conjunto_H, desviacion_conjunto_H, promedio_conjunto_H)
        # FO = sum(z) - lambda * cv_conjunto_N
        #sys.exit('También normalizar cv conjunto H y definir un nuevo lambda_ponderador inicial')
        FO = df_extremos['z'].sum() / len(df_extremos) - lambda_ponderador * cv_conjunto_H # Se divide por len para normalizar
    else:
        FO = (df_extremos['z'].sum() / len(df_extremos)) / cv_conjunto_H
    
    #if j % 400 == 0:
    #    print('F', j, df_extremos['z'].sum() / len(df_extremos) / cv_conjunto_H, df_extremos['z'].sum() / len(df_extremos), cv_conjunto_H)
    FO = float(FO)
    # 260119
    particion = [df_extremos['z'].sum() / len(df_extremos), cv_conjunto_H]
    return FO, df_extremos, particion

In [ ]:
"""
def traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes):
    if sizes == []:
        return conjunto_M, conjunto_N
    if sizes[0] + sizes[1] != len(conjunto_M) + len(conjunto_N):
        sys.exit('Error en sizes declarados')
    elif (sizes[0] != len(conjunto_M)) or (sizes[1] != len(conjunto_N)):
        delta = len(conjunto_M) - sizes[0]
        if delta > 0:
            #pasar delta elementos random de conjunto_M a conjunto_N
            elementos_a_pasar = random.sample(list(conjunto_M), delta)
            for elem in elementos_a_pasar:
                conjunto_M.remove(elem)
                conjunto_N.add(elem)
        else:
            #pasar delta elementos random de conjunto_N a conjunto_M
            elementos_a_pasar = random.sample(list(conjunto_N), -delta)
            for elem in elementos_a_pasar:
                conjunto_N.remove(elem)
                conjunto_M.add(elem)
    return conjunto_M, conjunto_N
"""

In [ ]:
def buscar_soportes_optimos(valor, N, df_extremos, conjunto_N, conjunto_M, lambda_ponderador, sizes = [], prints_general= False, n_randoms_iniciales = 0):
    
    #conjunto_M, conjunto_N = traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes)
    len_N_original = len(conjunto_N)
    
    # Uno a uno, reemplaza un punto del conjunto N por uno del conjunto M
    if prints_general:
        print('Evaluar cortar despues de un tiempo o si la FO no mejora más que alpha %')    
    df_FO = pd.DataFrame(columns = ['Iteracion', 'FO']) # Guarda los registros del valor de FO en cada iteración

    #sys.exit('[MODIFICAR] Conjunto N tambien debe incluir los soportes ACTIVOS. Con los soportes potenciales, hay que completar los ABIERTOS, pero los ACTIVOS deben permanecer en el conjunto N')
    FO = -float('inf')
    if n_randoms_iniciales > 0:
        for j in tqdm.tqdm(range(n_randoms_iniciales), desc='Random iniciales'):
            # Generar conjuntos N y M aleatorios
            conjunto_N_iter = set(random.sample(list(conjunto_N.union(conjunto_M)), len_N_original))
            conjunto_M_iter = conjunto_N.union(conjunto_M).difference(conjunto_N_iter)
            
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO: # Si es mejor, se actualiza
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                conjunto_N = conjunto_N_iter
                conjunto_M = conjunto_M_iter
                
    # inicialización
    FO, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
    i = 0
    df_registros = pd.DataFrame()
    while True:
        cambios = False
        #print('i', i)
        j = 0
        #df_n = pd.DataFrame('n': list(conjunto_N))
        #df_m = pd.DataFrame('m': list(conjunto_M))
        #df_iters = pd.DataFrame(itertools.product(df_n['n'], df_m['m']), columns=['n', 'm'])
        for n, m in tqdm.tqdm(itertools.product(conjunto_N, conjunto_M), total=len(conjunto_N)*len(conjunto_M)): # Para cada par (n,m) de los conjuntos N y M
            j += 1
            #print(n, m)
            
            # Se cambia el par n por m, de los conjuntos
            conjunto_N_iter = conjunto_N.copy()
            conjunto_N_iter.remove(n)
            conjunto_N_iter.add(m)
            
            conjunto_M_iter = conjunto_M.copy()
            conjunto_M_iter.remove(m)
            conjunto_M_iter.add(n)
            
            if len(conjunto_N_iter) != len_N_original:
                print(len(conjunto_N), len(conjunto_N_iter))
                
                sys.exit('Error en tamaño conjunto N iterado')
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO: # Si es mejor, break y se vuelven a recorrer todos los pares
                #print(f'Nueva FO: {round(FO_iter, 1)} > {round(FO, 1)}, con n={round(n, 0)} por m={round(m, 0)}')
                #print(f'Mejora de {round(FO_iter - FO, 2)}')
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                conjunto_N = conjunto_N_iter
                conjunto_M = conjunto_M_iter
                cambios = True
                pickle_act((f'conjuntosN2/{valor}_{N}_beta'), conjunto_N, 'save')
                pickle_act((f'conjuntosN2/respaldos/{valor}_{N}_beta'), conjunto_N, 'save')
                break
        
              
        df_FO_new = pd.DataFrame({'Iteracion': [i], 'FO': [FO], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]], 'n_iters': [j], 'prop_total': [j / (len(conjunto_N) * len(conjunto_M))]}) # Se guarda el registro
        
        if i % 10 == 0:
            display('df_FO_new', df_FO_new)
        df_FO = pd.concat([df_FO, df_FO_new])
        i += 1
        if not cambios:
            break
    
    return conjunto_N, conjunto_M, df_extremos, df_FO


In [ ]:
def buscar_soportes_optimos_v2(df_extremos, conjunto_N, conjunto_M, lambda_ponderador, sizes = [], prints_general= False): #260120
    # v2 itera todas las combinaciones hasta el final y eleige la mejor
    #conjunto_M, conjunto_N = traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes)
    len_N_original = len(conjunto_N)
    
    # Uno a uno, reemplaza un punto del conjunto N por uno del conjunto M
    if prints_general:
        print('Evaluar cortar despues de un tiempo o si la FO no mejora más que alpha %')    
    df_FO = pd.DataFrame(columns = ['Iteracion', 'FO']) # Guarda los registros del valor de FO en cada iteración

    #sys.exit('[MODIFICAR] Conjunto N tambien debe incluir los soportes ACTIVOS. Con los soportes potenciales, hay que completar los ABIERTOS, pero los ACTIVOS deben permanecer en el conjunto N')

    # inicialización
    FO, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # FUnción objetivo con conjuntos iniciales
    print('FO inicial', FO)
    i = 0
    while True:
        #cambios = False
        #print('i', i)
        j = 0
        FO_j = -float('inf')
        for n, m in tqdm.tqdm(itertools.product(conjunto_N, conjunto_M), total=len(conjunto_N)*len(conjunto_M)): # Para cada par (n,m) de los conjuntos N y M
            j += 1
            #print(n, m)
            
            # Se cambia el par n por m, de los conjuntos
            conjunto_N_iter = conjunto_N.copy()
            conjunto_N_iter.remove(n)
            conjunto_N_iter.add(m)
            
            conjunto_M_iter = conjunto_M.copy()
            conjunto_M_iter.remove(m)
            conjunto_M_iter.add(n)
            
            if len(conjunto_N_iter) != len_N_original:
                print(len(conjunto_N), len(conjunto_N_iter))
                
                sys.exit('Error en tamaño conjunto N iterado')
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO_j: # Si es mejor, break y se vuelven a recorrer todos los pares
                #print(f'Nueva FO: {round(FO_iter, 1)} > {round(FO, 1)}, con n={round(n, 0)} por m={round(m, 0)}')
                #print(f'Mejora de {round(FO_iter - FO, 2)}')
                FO_j = FO_iter
                mejor_tupla = (n, m)
        
        # Si es mejor que FO, se hace el mejor de todos los cambios
        if FO_j > FO:
            n, m = mejor_tupla
            # Se cambia el par n por m, de los conjuntos
            conjunto_N.remove(n)
            conjunto_N.add(m)
            conjunto_M.remove(m)
            conjunto_M.add(n)
            FO = FO_j
        else:
            break # Se encuentra mejor resultado
        
        df_FO_new = pd.DataFrame({'Iteracion': [i], 'FO': [FO], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]], 'n_iters': [j], 'prop_total': [j / (len(conjunto_N) * len(conjunto_M))]}) # Se guarda el registro
        
        #if i % 10 == 0:
        display('df_FO_new', df_FO_new)
        df_FO = pd.concat([df_FO, df_FO_new])
        i += 1
        #if not cambios:
        #    break
    
    return conjunto_N, conjunto_M, df_extremos, df_FO


In [ ]:
def buscar_soportes_optimos_v3(df_extremos, conjunto_N, conjunto_M, lambda_ponderador, sizes = [], prints_general= False, n_randoms_iniciales = 0):
    
    #conjunto_M, conjunto_N = traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes)
    len_N_original = len(conjunto_N)
    
    # Uno a uno, reemplaza un punto del conjunto N por uno del conjunto M
    if prints_general:
        print('Evaluar cortar despues de un tiempo o si la FO no mejora más que alpha %')    
    df_FO = pd.DataFrame(columns = ['Iteracion', 'FO']) # Guarda los registros del valor de FO en cada iteración

    #sys.exit('[MODIFICAR] Conjunto N tambien debe incluir los soportes ACTIVOS. Con los soportes potenciales, hay que completar los ABIERTOS, pero los ACTIVOS deben permanecer en el conjunto N')
    FO = -float('inf')
    if n_randoms_iniciales > 0:
        for j in tqdm.tqdm(range(n_randoms_iniciales), desc='Random iniciales'):
            # Generar conjuntos N y M aleatorios
            conjunto_N_iter = set(random.sample(list(conjunto_N.union(conjunto_M)), len_N_original))
            conjunto_M_iter = conjunto_N.union(conjunto_M).difference(conjunto_N_iter)
            
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO: # Si es mejor, se actualiza
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                conjunto_N = conjunto_N_iter
                conjunto_M = conjunto_M_iter
                
    # inicialización
    FO, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
    print('FO inicial', FO)
    i = 0
    df_registros = pd.DataFrame(columns = ['n', 'm', 'delta'])
    while True:
        cambios = False
        #print('i', i)
        #j = 0
        df_n = pd.DataFrame({'n': list(conjunto_N)})
        df_m = pd.DataFrame({'m': list(conjunto_M)})
        df_iters = df_n.merge(df_m, how='cross')
        df_iters = df_iters.merge(df_registros, on=['n', 'm'], how='left')
        df_iters['delta'] = df_iters['delta'].fillna(0)
        df_iters = df_iters.sort_values(by='delta', ascending=False).reset_index(drop=True) # Descendiente, para buscar un cambio positivo lo antes posible
        for j in range(len(df_iters)):
            n = df_iters.loc[j, 'n']
            m = df_iters.loc[j, 'm']
        #for n, m in tqdm.tqdm(itertools.product(conjunto_N, conjunto_M), total=len(conjunto_N)*len(conjunto_M)): # Para cada par (n,m) de los conjuntos N y M
            #j += 1
            #print(n, m)
            
            # Se cambia el par n por m, de los conjuntos
            conjunto_N_iter = conjunto_N.copy()
            conjunto_N_iter.remove(n)
            conjunto_N_iter.add(m)
            
            conjunto_M_iter = conjunto_M.copy()
            conjunto_M_iter.remove(m)
            conjunto_M_iter.add(n)
            
            if len(conjunto_N_iter) != len_N_original:
                print(len(conjunto_N), len(conjunto_N_iter))
                
                sys.exit('Error en tamaño conjunto N iterado')
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            delta = FO_iter - FO
            # busca (n, m) en df_registros y reeplaza delta si existe la tupla, si no, creala
            if ((df_registros['n'] == n) & (df_registros['m'] == m)).any():
                df_registros.loc[(df_registros['n'] == n) & (df_registros['m'] == m), 'delta'] = delta
            else:
                df_registros = pd.concat([df_registros, pd.DataFrame({'n': [n], 'm': [m], 'delta': [delta]})])
                
            # Se evalúa si es mejor
            if FO_iter > FO: # Si es mejor, break y se vuelven a recorrer todos los pares
                #print(f'Nueva FO: {round(FO_iter, 1)} > {round(FO, 1)}, con n={round(n, 0)} por m={round(m, 0)}')
                #print(f'Mejora de {round(FO_iter - FO, 2)}')
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                conjunto_N = conjunto_N_iter
                conjunto_M = conjunto_M_iter
                cambios = True
                break
        
              
        df_FO_new = pd.DataFrame({'Iteracion': [i], 'FO': [FO], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]], 'n_iters': [j], 'prop_total': [j / (len(conjunto_N) * len(conjunto_M))]}) # Se guarda el registro
        
        if i % 1 == 0:
            display('df_FO_new', df_FO_new)
        df_FO = pd.concat([df_FO, df_FO_new])
        i += 1
        if not cambios:
            break
    
    return conjunto_N, conjunto_M, df_extremos, df_FO


In [ ]:
def buscar_soportes_optimos_v4(valor, N, df_extremos, conjunto_N, conjunto_M, lambda_ponderador, sizes = [], prints_general= False, n_randoms_iniciales = 0):
    
    #conjunto_M, conjunto_N = traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes)
    len_N_original = len(conjunto_N)
    
    # Uno a uno, reemplaza un punto del conjunto N por uno del conjunto M
    if prints_general:
        print('Evaluar cortar despues de un tiempo o si la FO no mejora más que alpha %')    
    df_FO = pd.DataFrame(columns = ['Iteracion', 'FO']) # Guarda los registros del valor de FO en cada iteración

    #sys.exit('[MODIFICAR] Conjunto N tambien debe incluir los soportes ACTIVOS. Con los soportes potenciales, hay que completar los ABIERTOS, pero los ACTIVOS deben permanecer en el conjunto N')
    FO = -float('inf')
    if n_randoms_iniciales > 0:
        for j in tqdm.tqdm(range(n_randoms_iniciales), desc='Random iniciales'):
            # Generar conjuntos N y M aleatorios
            conjunto_N_iter = set(random.sample(list(conjunto_N.union(conjunto_M)), len_N_original))
            conjunto_M_iter = conjunto_N.union(conjunto_M).difference(conjunto_N_iter)
            
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO: # Si es mejor, se actualiza
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                conjunto_N = conjunto_N_iter
                conjunto_M = conjunto_M_iter
                
    # inicialización
    FO, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
    print('FO inicial', FO)
    i = 0
    df_registros = pd.DataFrame()
    while True:
        cambios = False
        #print('i', i)
        j = 0
        lista_N, lista_M = list(conjunto_N), list(conjunto_M)
        #df_n = pd.DataFrame('n': list(conjunto_N))
        #df_m = pd.DataFrame('m': list(conjunto_M))
        #df_iters = pd.DataFrame(itertools.product(df_n['n'], df_m['m']), columns=['n', 'm'])
        for m, n in tqdm.tqdm(itertools.product(lista_M, lista_N), total=len(lista_M)*len(lista_N)): # Para cada par (n,m) de los conjuntos N y M
            j += 1
            #print(n, m)
            
            # Se cambia el par n por m, de los conjuntos
            lista_N_iter = lista_N[:] # Copia
            lista_N_iter.remove(n)
            lista_N_iter.append(m)
            
            lista_M_iter = lista_M[:] # Copia
            lista_M_iter.remove(m)
            lista_M_iter.append(n)
            
            conjunto_N_iter = set(lista_N_iter)
            conjunto_M_iter = set(lista_M_iter)
            
            if len(lista_N_iter) != len_N_original:
                print(len(conjunto_N), len(conjunto_N_iter))
                
                sys.exit('Error en tamaño conjunto N iterado')
            FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
            
            if FO_iter > FO: # Si es mejor, break y se vuelven a recorrer todos los pares
                #print(f'Nueva FO: {round(FO_iter, 1)} > {round(FO, 1)}, con n={round(n, 0)} por m={round(m, 0)}')
                #print(f'Mejora de {round(FO_iter - FO, 2)}')
                FO = FO_iter
                df_extremos = df_extremos_iter
                particion_FO = particion_FO_iter
                
                # NUevo ordenamiento de listas
                # Para lista N, si la lista es [A | n(m) | B], donde A es todo lo que viene antes de n (reemplazado por m) y B despues, quiero dejar el orden [B | A | n(m)]
                pos_n = lista_N.index(n)
                lista_N = lista_N[pos_n + 1:] + lista_N[:pos_n] + [m]
                pos_m = lista_M.index(m)
                lista_M = lista_M[pos_m + 1:] + lista_M[:pos_m] + [n]
                
                #lista_N = lista_N_iter[:]
                #lista_M = lista_M_iter[:]
                # reordenar listas
                
                cambios = True
                conjunto_N = set(lista_N)
                conjunto_M = set(lista_M)
                pickle_act((f'conjuntosN22/{valor}_{N}_beta'), conjunto_N, 'save')
                break
        
              
        df_FO_new = pd.DataFrame({'Iteracion': [i], 'FO': [FO], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]], 'n_iters': [j], 'prop_total': [j / (len(conjunto_N) * len(conjunto_M))]}) # Se guarda el registro
        
        if i % 10 == 0:
            display('df_FO_new', df_FO_new)
        df_FO = pd.concat([df_FO, df_FO_new])
        i += 1
        if not cambios:
            break
    
    return conjunto_N, conjunto_M, df_extremos, df_FO


In [ ]:
def buscar_soportes_optimos_v5(df_extremos, conjunto_N, lambda_ponderador, sizes = [], prints_general= False, n_randoms_iniciales = 0):
    
    #conjunto_M, conjunto_N = traspasos_entre_conjuntos(conjunto_M, conjunto_N, sizes)
    #len_N_original = len(conjunto_N)
    
    # Uno a uno, reemplaza un punto del conjunto N por uno del conjunto M
    if prints_general:
        print('Evaluar cortar despues de un tiempo o si la FO no mejora más que alpha %')    
    #df_FO = pd.DataFrame(columns = ['Iteracion', 'FO']) # Guarda los registros del valor de FO en cada iteración

    #sys.exit('[MODIFICAR] Conjunto N tambien debe incluir los soportes ACTIVOS. Con los soportes potenciales, hay que completar los ABIERTOS, pero los ACTIVOS deben permanecer en el conjunto N')
    #FO = -float('inf')

    # inicialización
    print('conjuntoN1', conjunto_N)
    FO, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
    print('FO inicial', FO)
    return None


In [ ]:
def graficar_performance_FO(df_FO, graficar = False):
    if not graficar:
        return None
    # Graficar df_FO a doble eje (FO y n_iters vs iteracion)
    #display(f'df_FO {len(df_FO)}', df_FO)
    if len(df_FO) <= 1:
        print('FO sin Evolución \n\n\n')
        return None
    plt.figure(figsize = (21, 7))
    plt.plot(df_FO['Iteracion'], df_FO['FO'], linestyle='-', color='b', label='FO')
    plt.ylabel('FO')
    plt.xlabel('Iteración')
    if 'n_iters' in df_FO.columns:
        plt.twinx()
        plt.plot(df_FO['Iteracion'], df_FO['n_iters'], linestyle='-', color='r', label='n_iters')
        plt.ylabel('n_iters')
    plt.title('Evolución de la Función Objetivo (FO) y n_iters por Iteración')
    plt.legend()
    plt.grid()
    plt.show()
    
    print('Gráfico Biobjetivo') # 260119
    plt.figure(figsize = (21, 7))
    plt.plot(df_FO['Iteracion'], df_FO['FO1'], linestyle='-', color='b', label='FO1')
    plt.ylabel('FO1')
    plt.xlabel('Iteración')
    plt.twinx()
    plt.plot(df_FO['Iteracion'], df_FO['FO2'], linestyle='-', color='r', label='FO2')
    plt.ylabel('FO2')
    plt.title('Evolución de la Función Objetivo (FO) y n_iters por Iteración')
    plt.legend()
    plt.grid()
    plt.show()
    return None

In [ ]:
def graficar_soportes_all(df_0, conjunto_N, graficar = False, graficar_zoom = False):
        
    if graficar_zoom:
        df_0 = df_0.tail(100)
        minimo, maximo = df_0['Low'].min(), df_0['High'].max()
        plt.figure(figsize = (21, 7))
        plt.plot(df_0['DateTime'], df_0['Low'], linestyle='-', color='b', label='Low')
        plt.plot(df_0['DateTime'], df_0['High'], linestyle='-', color='g', label='High')
        for n in list(conjunto_N):
            if minimo <= n <= maximo:
                plt.axhline(y = n, color = 'r', linestyle = '--', alpha = 0.5)
        plt.xlabel('DateTime')
        plt.ylabel('Price')
        plt.title('Gráfico de Low en función de DateTime [Zoom]')
        plt.grid() # Agrega una cuadrícula
        plt.legend()
        plt.show()
        return None
    
    if not graficar:
        return None
        

    plt.figure(figsize = (21, 7))
    plt.plot(df_0['DateTime'], df_0['Low'], linestyle='-', color='b', label='Low')
    plt.plot(df_0['DateTime'], df_0['High'], linestyle='-', color='g', label='High')
    for n in list(conjunto_N):
        plt.axhline(y = n, color = 'r', linestyle = '--', alpha = 0.5)
    plt.xlabel('DateTime')
    plt.ylabel('Price')
    plt.title('Gráfico de Low en función de DateTime')
    plt.grid() # Agrega una cuadrícula
    plt.legend()
    plt.show()
    
    return None


In [ ]:
def generar_nuevas_oc(df_oc_all, conjunto_N, limite_low, limite_high, beta_low, beta_up, ultimo_cierre, delta_cambio_SL, delta_trigger, ultimo_periodo, delta_SL_LP):
    df_oc = pd.DataFrame()
    id_ = df_oc_all.shape[0] + 1
    n_distinct = set() # Por defecto vacío
    
    display('df_oc_all base inicial', df_oc_all.head(15))
    ids_abiertos = []
    if len(df_oc_all) > 0:
        ids_abiertos = df_oc_all[df_oc_all['Status'] == 'Abierta']['id'].tolist()
        #dfa1 = df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True)
        #display('A1', df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True))

    if ('Status' in df_oc_all.columns) and ('Monto Compra (lotes)' in df_oc_all.columns):
        df_rev = df_oc_all[(df_oc_all['Status'] == 'Activa') & (df_oc_all['Monto Compra (lotes)'].notna())]
        if len(df_rev) > 0:
            display('df_rev', df_rev)
            sys.exit('No pueden haber activas con monto de compra notna')
    
    
    if len(df_oc_all) > 0:
        ##display('A2', df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True))
        #dfa2 = df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True)
        # Si están activas, se desactivan
        df_oc_all['Status'] = np.where(df_oc_all['Status'] == 'Activa', 'Recien Inactiva', df_oc_all['Status'])
        n_distinct = set(df_oc_all[df_oc_all['Status'] == 'Recien Inactiva']['n'])
        #dfa3 = df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True)
        #display('A3', df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True))
    #display('df_oc_all base inicial 1', df_oc_all)
    
    #print('conjunto_N', conjunto_N)
    #display('df_oc_all en generar_nuevas_oc', df_oc_all)
        
    for n in list(conjunto_N):
        if n in n_distinct: # No se "reemplaza" un soporte que se mantiene
            # Volver a "Activar" solo si status no es "Abierta"
            df_oc_all['Status'] = np.where((df_oc_all['n'] == n) & (df_oc_all['Status'] != 'Abierta'), 'Activa', df_oc_all['Status'])
            # Si existe
            if len(df_oc_all[(df_oc_all['n'] == n) & (df_oc_all['Status'] == 'Activa')]) > 0:
                continue # No agregar nuevamente
        #print(n)
        if n < limite_low:
            beta = beta_low
        if n > limite_high:
            beta = beta_up
        else: # interpolación
            beta = beta_low + (beta_up - beta_low) * (n - limite_low) / (limite_high - limite_low)
        
        # Definir si es superior o inferior
        if n <= ultimo_cierre:
            tipo_oc = 'I'  # Inferior
            cambio_SL = None
            trigger = n # Se ejecuta la orden de compra en n
        else:
            tipo_oc = 'S'  # Superior
            cambio_SL = n * (1 + delta_cambio_SL / 100)
            trigger = cambio_SL * (1 + delta_trigger / 100)
        
        
        df_oc_new = pd.DataFrame({'id': [id_],
                                'tipo_oc': [tipo_oc],
                                'n': [n],
                                'beta': [beta], # Proporcion de la cuenta a comprar
                                'cambio_SL': [cambio_SL],
                                'trigger': [trigger],
                                'SL_LP': [n - delta_SL_LP],
                                'Status': ['Activa'],
                                'Monto Compra (lotes)': [np.nan],
                                'Valor_actual': [0]}) # Activa (esperando gatillar), Inactiva (fue reemplazada), Abierta (actualmente abierta), 'Cerrada' (se cerró la orden)
        df_oc = pd.concat([df_oc, df_oc_new])
        id_ += 1
        #print('beta para n =', n, ' es ', beta)

    #if len(df_oc_all) > 0:
        #dfa4 = df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True)
    #    display('A4', df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True))

    if len(df_oc) > 0:
        df_oc['ultimo_cierre'] = ultimo_cierre
        df_oc['periodo'] = ultimo_periodo
        
        #display(f'df_oc {len(conjunto_N)}', df_oc)
        df_oc = df_oc[['periodo', 'ultimo_cierre', 'id', 'tipo_oc', 'n', 'beta', 'cambio_SL', 'trigger', 'SL_LP', 'Status', 'Monto Compra (lotes)', 'Valor_actual']]
        df_oc_all = pd.concat([df_oc_all, df_oc]).reset_index(drop=True)
    df_oc_all['Status'] = np.where(df_oc_all['Status'] == 'Recien Inactiva', 'Inactiva', df_oc_all['Status'])
    #if len(df_oc_all) > 0:
        #dfa5 = df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True)
        #display('A5', df_oc_all[df_oc_all['Status'] == 'Abierta'].reset_index(drop = True))

    ids_activos = []
    if len(df_oc_all) > 0:
        ids_activos = df_oc_all[df_oc_all['Status'] == 'Activa']['id'].tolist()
        
    # Valida si algun id pasa de Abierto a Activo
    if any(id_ in ids_abiertos for id_ in ids_activos):
        # print esos ids
        print("Los siguientes ids pasaron de Abierta a Activa:", [id_ for id_ in ids_activos if id_ in ids_abiertos])
        #display('dfa1', dfa1)
        #display('dfa2', dfa2)
        #display('dfa3', dfa3)
        #display('dfa4', dfa4)
        #display('dfa5', dfa5)
        #dfa1.to_csv('dfa1.csv', index=False, sep = ';', decimal = ',')
        #dfa2.to_csv('dfa2.csv', index=False, sep = ';', decimal = ',')
        #dfa3.to_csv('dfa3.csv', index=False, sep = ';', decimal = ',')
        #dfa4.to_csv('dfa4.csv', index=False, sep = ';', decimal = ',')
        #dfa5.to_csv('dfa5.csv', index=False, sep = ';', decimal = ',')
        sys.exit("Entre 'df_oc_all base inicial' y 'df_oc_all base final', hay status que pasan de Abierta a Activa...eso no puede ocurrir")
    
    display('df_oc_all base final', df_oc_all.head(15))
    return df_oc_all

In [ ]:
def actualizar_stop_loss_ordenes_abiertas(df_oc_all, cambio_SL, valor, continuar_por_ahora):
    
    if continuar_por_ahora:
        return df_oc_all
    
    print('\n\n\n actualizar_stop_loss_ordenes_abiertas')
    display('df_oc_all', df_oc_all)
    print('cambio_SL', cambio_SL)
    sys.exit('Desarrollar, solo para operaciones abiertas y si cambio_SL es mayor al precio de compra')

In [ ]:
# Prueba
trigger = 1202.557551
cuenta_size = 3000
beta = 4.99169561
monto_compra = 0.1
grano_lote = 0.1
U = 1

trigger - cuenta_size * (beta / 100) / (monto_compra * U)
sl_lp_new = 0
(cuenta_size * (beta / 100) / ((trigger - sl_lp_new) * U)) // grano_lote * grano_lote

In [ ]:
# Prueba
monto_compra = (cuenta_size * (beta / 100) / ((trigger - sl_lp_new) * U)) // grano_lote * grano_lote
sl_lp_new = trigger - cuenta_size * (beta / 100) / (monto_compra * U)
sl_lp_new

In [ ]:
def ejecutar_orden_de_compra(df_oc_all, cuenta_size, beta, trigger, sl_lp, valor, _id, minimo_lotaje = False):
    if valor not in ['BTCUSD', 'ETHUSD']:
        sys.exit('Solo desarrollado para BTCUSD por ahora...cambiar lógica de grano lote y U')
        
    granos_lotes = {'BTCUSD': 0.01, 'ETHUSD': 0.1}
    unidades = {'BTCUSD': 1, 'ETHUSD': 1}
    grano_lote = granos_lotes[valor] # tamaño mínimo de lote para BTCUSD
    U = unidades[valor] # Unidades de lote: 1 lote = 1 BTCUSD

    monto_compra = (cuenta_size * (beta / 100) / ((trigger - sl_lp) * U)) // grano_lote * grano_lote
    monto_compra # Lotes
    e1, e2, e3, e4 = False, False, False, False
    # Entonces, se ajusta  el SL_LP 
    if minimo_lotaje:
        monto_compra = grano_lote
        sl_lp_new = trigger - cuenta_size * (beta / 100) / (monto_compra * U)
        e1 = True

    elif monto_compra == 0:
        sl_lp_new = sl_lp
        e2 = True
    else:
        sl_lp_new = trigger - cuenta_size * (beta / 100) / (monto_compra * U)
        e3 = True
    # Aunque sean operaciones de mínimo lotaje
    if sl_lp_new < 0:
        sl_lp_new = 0
        monto_compra = (cuenta_size * (beta / 100) / ((trigger - sl_lp_new) * U)) // grano_lote * grano_lote
        e4 = True

    # Validacion
    if (sl_lp_new > 0) and (abs(monto_compra - (cuenta_size * (beta / 100) / ((trigger - sl_lp_new) * U))) > 0.001): # Si el sl LP era negativo, entonces no se cumple la regla 
        print('monto_compra', monto_compra)
        #print('sl_lp_new', sl_lp_new)
        print('cuenta_size', cuenta_size)
        print('beta', beta)
        print('trigger', trigger)
        print('sl_lp_new', sl_lp_new)
        print('U', U)
        print('e1, e2, e3, e4', e1, e2, e3, e4)
        sys.exit('Error en cálculo de monto compra y SL_LP_new')

    # Se agregan / actualizan los campos

    df_oc_all.loc[df_oc_all['id'] == _id, 'Monto Compra (lotes)'] = monto_compra
    df_oc_all.loc[df_oc_all['id'] == _id, 'SL_LP'] = sl_lp_new
    df_oc_all.loc[df_oc_all['id'] == _id, 'Status'] = 'Abierta' 
    df_oc_all.loc[df_oc_all['id'] == _id, 'Valor_actual'] = 0 # Nada cuando se crea
    
    return df_oc_all

In [ ]:
def nuevo_optimizador_0(N, df_extremos, conjunto_N, lambda_ponderador, ordenes_activas = [], M = 100, max_iters = 200):
    df_FO = pd.DataFrame() # Guarda los registros del valor de FO en cada iteración
    #FO = -float('inf')
    #print('conjuntoN2A', conjunto_N)
    delta = N - len(ordenes_activas) - len(conjunto_N) # Cantidad de soportes que se pueden elegir libremente, sin contar los soportes activos (ordenes activas)
    delta2 = N - len(ordenes_activas)
    print('delta y delta2', delta, delta2, [N, len(ordenes_activas), len(conjunto_N)])
    #delta y delta2 0 36 [36, 0, 36]
    #delta y delta2 0 36 [36, 0, 36]
    
    if delta2 < 0:
        print('Cantidad de ordenes activas es mayor a N, revisar')
        sys.exit()
    
    p_min, p_max = df_extremos['Low'].min(), df_extremos['Low'].max()
    #if prints_general:
    print('p_min y p_max', p_min, p_max)

    # Se debe entregar conjunto_N y sizes
    #conjunto_M = set(np.linspace(p_min, p_max, M).tolist()) # M precios equidistantes entre p_min y p_max (potenciales soportes)
    if delta >= 0:
        conjunto_N = conjunto_N.union(set(np.random.uniform(p_min, p_max, delta).tolist())) # delta con distribución uniforme
    elif delta2 > 0: # Se crea de nuevo, omitiendo conjunto_N input
        conjunto_N = set(np.random.uniform(p_min, p_max, delta2).tolist()) # delta con distribución uniforme
    #len(conjunto_N)

    conjunto_N = conjunto_N.union(set(ordenes_activas)) # Se agregan las ordenes activas al conjunto N, ya que deben ser soportes fijos
    #print('conjuntoN2B', conjunto_N)
    if len(conjunto_N) != N: #Validador
        print(len(conjunto_N), N)
        sys.exit('Error en tamaño conjunto N')
        
    lista_N = list(conjunto_N)
    lista_N.sort()
    dic_N = {i: n for i, n in enumerate(lista_N)}

    casos_moviles = list(dic_N.keys()) # inicializacion
    for j in range(max_iters):
        lista_N = list(dic_N.values())
        conjunto_N = set(lista_N)
        #print('conjuntoN2C', conjunto_N)
        FO_base, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
        #print('INICIAL')
        #print(f'FO {j}', FO_base)
        #print(sorted(lista_N))
        #sys.exit('Revisión de FOs')
        mejora = False
        for i in tqdm.tqdm(casos_moviles):
            cota_inferior = dic_N[i - 1] if i - 1 in dic_N else p_min
            cota_superior = dic_N[i + 1] if i + 1 in dic_N else p_max
            # Ahora genera un random uniforma de M casos entre cota inferior y superior
            casos_random = np.random.uniform(cota_inferior, cota_superior, M)

            for caso in casos_random:
                lista_N_iter = lista_N[:]
                lista_N_iter.remove(dic_N[i])
                lista_N_iter.append(caso)
                conjunto_N_iter = set(lista_N_iter)
                FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
                
                if FO_iter > FO_base: # Si es mejor, break y se vuelven a recorrer todos los pares
                    mejora = True
                    FO_base = FO_iter
                    i_change = i
                    nuevo_value = caso
                    df_extremos = df_extremos_iter.copy()
                    particion_FO = particion_FO_iter[:]
                    
        
        if not mejora:
            if len(casos_moviles) == len(dic_N): # Si ya se probaron todos los cass. Exit
                break
            # SI no, se amplían los casos móviles
            print('\n\n\n Se amplían los casos móviles a todos \n\n\n')
            casos_moviles = list(dic_N.keys()) # inicializacion
        else:
            # Hacer el cambio
            dic_N[i_change] = nuevo_value
            print(f'Nuevo FO: {notacion_cientifica(FO_base, 4)}, con cambio en i_change = {i_change}, n={round(lista_N[i_change], 0)} por caso={round(nuevo_value, 2)}')
            casos_moviles = [i_change - 1, i_change, i_change + 1] # Solo los vecinos del cambio, para la próxima iteración
            casos_moviles = [i_change - 1, i_change + 1] # Solo los vecinos del cambio, para la próxima iteración
        for caso in casos_moviles:
            if caso < 0 or caso >= len(dic_N):
                casos_moviles.remove(caso)
        
        df_FO_new = pd.DataFrame({'Iteracion': [j], 'FO': [FO_base], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]]}) # Se guarda el registro
        df_FO = pd.concat([df_FO, df_FO_new])
        
    return conjunto_N, df_extremos, df_FO



In [ ]:
def nuevo_optimizador_1(N, df_extremos, conjunto_N, lambda_ponderador, ordenes_activas = [], M = 100, max_iters = 1000, prueba_cercanos = False):
    df_FO = pd.DataFrame() # Guarda los registros del valor de FO en cada iteración
    #FO = -float('inf')
    #print('conjuntoN2A', conjunto_N)
    delta = N - len(ordenes_activas) - len(conjunto_N) # Cantidad de soportes que se pueden elegir libremente, sin contar los soportes activos (ordenes activas)
    delta2 = N - len(ordenes_activas)
    print('delta y delta2', delta, delta2, [N, len(ordenes_activas), len(conjunto_N)])
    #delta y delta2 0 36 [36, 0, 36]
    #delta y delta2 0 36 [36, 0, 36]
    
    if delta2 < 0:
        print('Cantidad de ordenes activas es mayor a N, revisar')
        sys.exit()
    
    p_min, p_max = df_extremos['Low'].min(), df_extremos['Low'].max()
    #if prints_general:
    print('p_min y p_max', p_min, p_max)

    # Se debe entregar conjunto_N y sizes
    #conjunto_M = set(np.linspace(p_min, p_max, M).tolist()) # M precios equidistantes entre p_min y p_max (potenciales soportes)
    if delta >= 0:
        conjunto_N = conjunto_N.union(set(np.random.uniform(p_min, p_max, delta).tolist())) # delta con distribución uniforme
    elif delta2 > 0: # Se crea de nuevo, omitiendo conjunto_N input
        conjunto_N = set(np.random.uniform(p_min, p_max, delta2).tolist()) # delta con distribución uniforme
    #len(conjunto_N)

    conjunto_N = conjunto_N.union(set(ordenes_activas)) # Se agregan las ordenes activas al conjunto N, ya que deben ser soportes fijos
    #print('conjuntoN2B', conjunto_N)
    if len(conjunto_N) != N: #Validador
        print(len(conjunto_N), N)
        sys.exit('Error en tamaño conjunto N')
        
    lista_N = list(conjunto_N)
    lista_N.sort()
    dic_N = {i: n for i, n in enumerate(lista_N)}

    casos_moviles = list(dic_N.keys()) # inicializacion
    for j in range(max_iters):
        lista_N = list(dic_N.values())
        conjunto_N = set(lista_N)
        #print('conjuntoN2C', conjunto_N)
        FO_base, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
        #print('INICIAL')
        #print(f'FO {j}', notacion_cientifica(FO_base, 4),)
        print('CASOS MOVILES', casos_moviles)
        #print(sorted(lista_N))
        #sys.exit('Revisión de FOs')
        mejora = False
        for i in tqdm.tqdm(casos_moviles):
            cota_inferior = dic_N[i - 1] if i - 1 in dic_N else p_min
            cota_superior = dic_N[i + 1] if i + 1 in dic_N else p_max
            # Ahora genera un random uniforma de M casos entre cota inferior y superior
            casos_random = np.random.uniform(cota_inferior, cota_superior, M)

            for caso in casos_random:
                lista_N_iter = lista_N[:]
                lista_N_iter.remove(dic_N[i])
                lista_N_iter.append(caso)
                conjunto_N_iter = set(lista_N_iter)
                FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
                
                if FO_iter > FO_base: # Si es mejor, break y se vuelven a recorrer todos los pares
                    mejora = True
                    FO_base = FO_iter
                    i_change = i
                    nuevo_value = caso
                    df_extremos = df_extremos_iter.copy()
                    particion_FO = particion_FO_iter[:]
                    break
            
            if mejora:
                break
                       
        if not mejora:
            if len(casos_moviles) == len(dic_N): # Si ya se probaron todos los cass. Exit
                break
            # SI no, se amplían los casos móviles
            print('\n\n\n Se amplían los casos móviles a todos \n\n\n')
            casos_moviles = list(dic_N.keys()) # inicializacion
        else:
            # Hacer el cambio
            dic_N[i_change] = nuevo_value
            print(f'Nuevo FO {j}: {notacion_cientifica(FO_base, 4)}, con cambio en i_change = {i_change}, n={round(lista_N[i_change], 0)} por caso={round(nuevo_value, 2)}')
            
            #sys.exit('Aquií hacer el cambio para que sean todos, partiendo por los que están declarados')
            if prueba_cercanos:
                casos_moviles_base = [i_change - 1, i_change + 1, i_change] # Solo los vecinos del cambio, para la próxima iteración
                for c in casos_moviles_base:
                    if c in casos_moviles:
                        casos_moviles.remove(c)
                
                random.shuffle(casos_moviles) # Los demás casos se ordenan de forma random

                casos_moviles = casos_moviles_base[:] + casos_moviles[:]
            else:
                random.shuffle(casos_moviles)
            
            #casos_moviles = [i_change - 1, i_change + 1] # Solo los vecinos del cambio, para la próxima iteración
        for caso in casos_moviles:
            if caso < 0 or caso >= len(dic_N):
                casos_moviles.remove(caso)
        
        df_FO_new = pd.DataFrame({'Iteracion': [j], 'FO': [FO_base], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]]}) # Se guarda el registro
        df_FO = pd.concat([df_FO, df_FO_new])
        
    return conjunto_N, df_extremos, df_FO



In [ ]:
a = np.random.uniform(0, 100, 10)
a.sort()
a

In [ ]:
# df de len 10 con cols 'caso' y 'FO_iter'. Numeros random
df_plot = pd.DataFrame({'caso': [1, 2, 3, 4], 'FO_iter': [5, 10, 6, 2]})


caso = df_plot.loc[df_plot['FO_iter'].idxmax(), 'caso']
FO_iter = df_plot['FO_iter'].max()

float(caso), FO_iter

In [ ]:
"""
# plot de y=x**2 con puntos y lineas
x = np.linspace(-10, 10, 100)
y = x**2
plt.figure(figsize=(10, 5))
plt.plot(x, y, label='y = x^2')
plt.scatter(x, x**2, color='red', label='Puntos aleatorios')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Gráfico de y = x^2 con puntos aleatorios')
plt.legend()
plt.grid()
plt.show()
"""

In [ ]:


def evaluar_crecimiento_decrecimiento(df_plot, metrica = 'y'):
    
    df_plot = df_plot.reset_index(drop = True)
    crec, decrec = True, False
    for i in range(1, len(df_plot)):
        a = df_plot[metrica][i] - df_plot[metrica][i - 1]
        if a > 0 and decrec:
            return False
        if a < 0 and crec:
            decrec = True
            crec = False
    return True
    


In [ ]:
def nuevo_optimizador_2(N, df_extremos, conjunto_N, lambda_ponderador, ordenes_activas = [], M = 100, max_iters = 1000, prueba_cercanos = False, delta_inicial = 10 ** -4, factor_reduccion = 0.9):
    
    # delta_inicial = 10 ** -3: AL menos debe mejorar un 0.1% (10E-3) para considerar que hay una mejora, sino se sigue buscando en la misma iteración con otros casos random. Esto evita hacer cambios por mejoras muy pequeñas, que podrían ser ruido.
    prueba260417 = True
    print('max_iters', max_iters)
    #if prueba260417:
    #    max_iters = 1
    
    
    df_FO = pd.DataFrame() # Guarda los registros del valor de FO en cada iteración
    #FO = -float('inf')
    #print('conjuntoN2A', conjunto_N)
    delta = N - len(ordenes_activas) - len(conjunto_N) # Cantidad de soportes que se pueden elegir libremente, sin contar los soportes activos (ordenes activas)
    delta2 = N - len(ordenes_activas)
    print('delta y delta2', delta, delta2, [N, len(ordenes_activas), len(conjunto_N)])
    #delta y delta2 0 36 [36, 0, 36]
    #delta y delta2 0 36 [36, 0, 36]
    
    if delta2 < 0:
        print('Cantidad de ordenes activas es mayor a N, revisar')
        sys.exit()
    
    p_min, p_max = df_extremos['Low'].min(), df_extremos['Low'].max()
    #if prints_general:
    print('p_min y p_max', p_min, p_max)

    # Se debe entregar conjunto_N y sizes
    #conjunto_M = set(np.linspace(p_min, p_max, M).tolist()) # M precios equidistantes entre p_min y p_max (potenciales soportes)
    if delta >= 0:
        conjunto_N = conjunto_N.union(set(np.random.uniform(p_min, p_max, delta).tolist())) # delta con distribución uniforme
    elif delta2 > 0: # Se crea de nuevo, omitiendo conjunto_N input
        conjunto_N = set(np.random.uniform(p_min, p_max, delta2).tolist()) # delta con distribución uniforme
    #len(conjunto_N)

    conjunto_N = conjunto_N.union(set(ordenes_activas)) # Se agregan las ordenes activas al conjunto N, ya que deben ser soportes fijos
    #print('conjuntoN2B', conjunto_N)
    if len(conjunto_N) != N: #Validador
        print(len(conjunto_N), N)
        sys.exit('Error en tamaño conjunto N')
        
    lista_N = list(conjunto_N)
    lista_N.sort()
    dic_N = {i: n for i, n in enumerate(lista_N)}

    casos_moviles = list(dic_N.keys()) # inicializacion
    for j in range(max_iters):
        lista_N = list(dic_N.values())
        conjunto_N = set(lista_N)
        if len(conjunto_N) != N:
            print(len(conjunto_N), N)
            print(dic_N.keys())
            print(lista_N)
            print(conjunto_N)
            display('df_plot', df_plot)
            sys.exit('PRUEBA 260529: Error en tamaño conjunto N en iteración')
        #else:
        #    print(f'Iteración {j}, tamaño conjunto N correcto: {len(conjunto_N)}')
        #print('conjuntoN2C', conjunto_N)
        FO_base, df_extremos, particion_FO = calcular_FO(df_extremos, conjunto_N, lambda_ponderador) # Función objetivo con conjuntos iniciales
        #print('INICIAL')
        #print(f'FO {j}', notacion_cientifica(FO_base, 4),)
        #print('CASOS MOVILES', casos_moviles)
        #print(sorted(lista_N))
        #sys.exit('Revisión de FOs')
        mejora = False
        for i in tqdm.tqdm(casos_moviles):
            #print('i', i)
            cota_inferior = dic_N[i - 1] if i - 1 in dic_N else p_min
            cota_superior = dic_N[i + 1] if i + 1 in dic_N else p_max
            # Ahora genera un random uniforma de M casos entre cota inferior y superior
            # casos_random debe ser una particion equidistante
            if prueba260417:
                casos_random = np.linspace(cota_inferior, cota_superior, M)
            else:
                casos_random = np.random.uniform(cota_inferior, cota_superior, M)
                casos_random.sort() # ordenamiento

            # Solucion propuesta 260529 para que no ocurra el problema de borde (que se repitan dos soportes/resistencias en la lista)
            casos_random = casos_random[1:-1] # Se eliminan los extremos, para evitar problemas de borde que podrían generar mejoras artificiales por salir del rango de búsqueda
            df_plot = pd.DataFrame()
            for caso in casos_random:
                lista_N_iter = lista_N[:]
                lista_N_iter.remove(dic_N[i])
                lista_N_iter.append(caso)
                conjunto_N_iter = set(lista_N_iter)
                FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador) # Se evalúa la función objetivo
                df_plot_new = pd.DataFrame({'caso': [caso], 'FO_iter': [FO_iter]})
                df_plot = pd.concat([df_plot, df_plot_new])
                
                """
                if prueba260417:
                    continue # Nunca break para ver graficos
                if (FO_iter - FO_base) / FO_base > delta_inicial: # Si es mejor, break y se vuelven a recorrer todos los pares
                    mejora = True
                    FO_base = FO_iter
                    i_change = i
                    nuevo_value = caso
                    df_extremos = df_extremos_iter.copy()
                    particion_FO = particion_FO_iter[:]
                    delta_inicial *= factor_reduccion # Se reduce el delta para la próxima iteración, buscando mejoras más pequeñas a medida que avanza la optimización
                    break
                """
            
            # Plotear
            if prueba260417:
                if len(dic_N) != N:
                    print(len(dic_N), N)
                    sys.exit('PRUEBA 260529 V3: Error en tamaño conjunto N en iteración')
                
                df_plot = df_plot.reset_index(drop = True)
                cumplen_logica = evaluar_crecimiento_decrecimiento(df_plot, metrica = 'FO_iter')
                if cumplen_logica:
                    #print('SI cumplen_logica')
                    # Aproximar a una función cuadrática a * x^2 + b * x + c, y verificar que a < 0 para confirmar la forma de U invertida
                    coeficientes = np.polyfit(df_plot['caso'], df_plot['FO_iter'], 2)
                    a, b, c = coeficientes
                    caso = -b / (2 * a) # El caso óptimo según la parábola
                    # Cálculo explícito en ese punto
                   
                    lista_N_iter = lista_N[:]
                    lista_N_iter.remove(dic_N[i])
                    lista_N_iter.append(caso)
                    conjunto_N_iter = set(lista_N_iter)
                    FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador)
                    
                else:
                    #print('NO cumplen_logica')
                    # caso es el punto más alto encontrado
                    #print('IDMAX (el supuesto es que si falla, idmax es 0)', df_plot['FO_iter'].idxmax())
                    caso = float(df_plot.loc[df_plot['FO_iter'].idxmax(), 'caso'])
                    FO_iter = float(df_plot['FO_iter'].max())
                # Cálculo explícito en ese punto
                #lista_N_iter = lista_N[:]
                #lista_N_iter.remove(dic_N[i])
                #lista_N_iter.append(caso)
                #conjunto_N_iter = set(lista_N_iter)
                #FO_iter, df_extremos_iter, particion_FO_iter = calcular_FO(df_extremos, conjunto_N_iter, lambda_ponderador)
                
                
            #print('Caso', caso, 'FO iter', FO_iter)
            if (FO_iter - FO_base) / FO_base > delta_inicial: # Si es mejor, break y se vuelven a recorrer todos los pares
                print('Factor de mejora', (FO_iter - FO_base) / FO_base, 'minimo', delta_inicial)
                mejora = True
                FO_base = FO_iter
                i_change = i
                nuevo_value = caso
                df_extremos = df_extremos_iter.copy()
                particion_FO = particion_FO_iter[:]
                #delta_inicial *= factor_reduccion # Se reduce el delta para la próxima iteración, buscando mejoras más pequeñas a medida que avanza la optimización
                #print('Mejora encontrada con caso', caso, 'FO base', FO_base)
            
                """
                plt.figure(figsize = (21, 7))
                # plot con puntos y lineas
                plt.scatter(df_plot['caso'], df_plot['FO_iter'], color='b', label='FO_iter')
                plt.plot(df_plot['caso'], df_plot['FO_iter'], linestyle='-', color='b', label='FO_iter')
                plt.axhline(y = FO_base, color = 'r', linestyle = '--', alpha = 0.5, label = 'FO_base')
                if cumplen_logica:
                    x_fit = np.linspace(df_plot['caso'].min(), df_plot['caso'].max(), 100)
                    y_fit = a * x_fit**2 + b * x_fit + c
                    plt.plot(x_fit, y_fit, linestyle='--', color='g', label='Ajuste cuadrático')
                plt.legend()
                plt.show()
                """
                    
                #print('Cumplen lógica de crecer y después decrecer?', cumplen_logica)

                # Si los puntos cumplen con la lógica de crecer y despues decrecer
                #cumplen_logica = False
            
            if len(dic_N) != N:
                print(len(dic_N), N)
                sys.exit('PRUEBA 260529 V2: Error en tamaño conjunto N en iteración')
                
            if mejora:
                break
                       
        if not mejora:
            if len(casos_moviles) == len(dic_N): # Si ya se probaron todos los casos. Exit
                break # No hay resultados que mejoren la solución actual
                #print('nuevo delta_inicial', delta_inicial)
            # SI no, se amplían los casos móviles
            print('\n\n\n Se amplían los casos móviles a todos \n\n\n')
            casos_moviles = list(dic_N.keys()) # inicializacion
        else:
            if len(dic_N) != N:
                print(len(dic_N), N)
                sys.exit('PRUEBA 260529 V3: Error en tamaño conjunto N en iteración')
            # Hacer el cambio
            dic_N[i_change] = nuevo_value
            print(f'Nuevo FO {j}: {notacion_cientifica(FO_base, 4)} | particion = [{notacion_cientifica(particion_FO[0], 4)}, {notacion_cientifica(particion_FO[1], 4)}] | con cambio en i_change = {i_change}, n={round(lista_N[i_change], 0)} por caso={round(nuevo_value, 2)}')

            if len(dic_N) != N:
                print(len(dic_N), N)
                sys.exit('PRUEBA 260529 V4: Error en tamaño conjunto N en iteración')
            #sys.exit('Aquií hacer el cambio para que sean todos, partiendo por los que están declarados')
            if prueba_cercanos:
                casos_moviles_base = [i_change - 1, i_change + 1, i_change] # Solo los vecinos del cambio, para la próxima iteración
                for c in casos_moviles_base:
                    if c in casos_moviles:
                        casos_moviles.remove(c)
                
                random.shuffle(casos_moviles) # Los demás casos se ordenan de forma random

                casos_moviles = casos_moviles_base[:] + casos_moviles[:]
            else:
                random.shuffle(casos_moviles)
            
            #casos_moviles = [i_change - 1, i_change + 1] # Solo los vecinos del cambio, para la próxima iteración
        for caso in casos_moviles:
            if caso < 0 or caso >= len(dic_N):
                casos_moviles.remove(caso)
        
        df_FO_new = pd.DataFrame({'Iteracion': [j], 'FO': [FO_base], 'FO1': [particion_FO[0]], 'FO2': [particion_FO[1]], 'ratio': [particion_FO[0] / particion_FO[1]]}) # Se guarda el registro
        df_FO = pd.concat([df_FO, df_FO_new])
        
    return conjunto_N, df_extremos, df_FO



# Algorítmo

In [ ]:
# Que viene después?

# 0. Agregar una nueva vela al df y hacer lo siguiente
# 1: Recalcular soportes y REEMPLAZAR los que estaban (esto puede hacerse cada x velas, no en todas) 
    # 1a. Crear ordenes de compra, bajo el precio inicial, en los soportes N (siempre)
    # 1b. Crear OC arriba del precio, siempre y cuando el precio máximo de la nueva vela, active el umbral
# 2: Evaluar si se ejecuta una OC abajo (toque de soporte) y/o una OC arriba (alcanza el trigger [gatillante]) > Ejecutarla y guardarla en una BD
#     Incluir en la BD el $ en cuenta, para calcular posteriormente pérdida o ganancia
# 3: Generar cambios de SL, si se gatilla una OC arriba
# 4: Evaluar si hay cierres de operación y cerrar en caso de que existan
# 5: Con RLM, y gradient descent (70% train - 30% test), optimizar los parámetros declarados arriba

# ** En todos los casos, ir guardando los registros

## Parámetros iniciales

In [ ]:
#valor = 'BTCUSD'
#valor = 'ETHUSD'
#paso = 24 * 14 # cada 14 días

In [ ]:
display_0 = True # Muestra df head inicial y el gráfico de velas completo
display_1 = False #True # Gráfico de velas, cada vez que una vela se agrega
graficar_extremos = True
graficar_perf_FO = True
graficar_soportes = True
graficar_zoom = True

## Parámetros sensibilizados por grad descent

In [ ]:
from Transversal import n_sizes

In [ ]:
n_sizes

In [ ]:
# Parámetros (PARS)

# T: Cantidad de días previos para selección de la ventana
T = 60 # 60 días
# k: ponderador para mínimos de los días futuros: P = delta pasado + k * delta futuro
k = 1
# n: exp de ponderación por tiempo: Ponderación por tiempo: w_i = exp(-n*(T-i)), i = 1,...,T / o puede ser w_i = A * t^n 
n = 1.3
# M: Cantidad de "soportes tentativos" en total (particiones)
M = 30

# lambda: ponderador para FO: Calidad resistencias - lambda * desv(h) / prom(h) ...h es la separación entre resistencias h_i = r_i - r_(i-1), para todo i = 2,...,N
lambda_ponderador = 1 / 500
# delta trigger (en %), sobre el cambio SL
#delta_trigger = 1
# delta cambio stop loss (en %)
#delta_cambio_SL = 1

# Que % (en %) máximo de la cuenta se arriesga a mu - 2 desv estandar (beta low) y a mu + 2 desv estandar (beta up)}
# SOlo aplica en versión lotaje_minimo = False
beta_low = 4.5
beta_up = 4

# Cuantas desv estandar abajo debe estar el SL de largo plazo
#z_sl_lp = 1

# Recordar que el orden, de arriba hacia abajo es delta trigger / delta cambio SL / soporte

In [ ]:
#print('Cambiar a estrategia con operaciones de minimo lotaje hasta llegar a aprox \n10k usd...sl de largo plazo debe ser el menor posible para comprar por ejemplo 0.01 BTCUSD')
#sys.exit()

## Ejecución

In [ ]:
#def leer_lista_N(valor, N):
#    lista_N = list(pickle_act(f'conjuntosN2/{valor}_{N}'))
#    lista_N = [round(n, 2) for n in lista_N]
#    return lista_N

In [ ]:
#valor = "ETHUSD"
#N = n_sizes[valor] # cantidad de soportes y resistencias a considerar
#lista_N = leer_lista_N(valor, N)
#lista_N.sort()
#lista_N

In [ ]:
cuenta_size = 3000 # Inicial, en USD
activar = True # Activar! Desactivado mientras solo para pruebas

In [ ]:
prints_generales = True
display_1 = True

Inicialmente, se lee el df completo y se gráfica el comportamiento histórico con velas

In [ ]:
#prueba260331 = True
prueba260401 = False

In [ ]:
valores = ['BTCUSD', 'ETHUSD', 'TSLA', 'GOOGL', 'NVDA', 'AMZN']
if prueba260401:
    valores = ['ETHUSD']

#valores = ['TSLA']

In [ ]:
import datetime

In [ ]:
df_valores = pd.DataFrame()
for valor in valores:
    N = n_sizes[valor]
    print(valor, N)
    if f'{valor}_{N}_beta.pkl' in os.listdir('conjuntosN2/'):
        conjunto_N_preliminar = pickle_act(f'conjuntosN2/{valor}_{N}_beta')   
        #print(valor, conjunto_N_preliminar)
        # mostrar fecha de modificacion del archivo
        fecha_modificacion = os.path.getmtime(f'conjuntosN2/{valor}_{N}_beta.pkl')
        fecha_modificacion = datetime.datetime.fromtimestamp(fecha_modificacion)
        print(f'Fecha de modificación del archivo: {fecha_modificacion}')
        
    else:
        # una fecha antigua
        fecha_modificacion = datetime.datetime(2000, 1, 1)
    df_valores_new = pd.DataFrame({'Valor': [valor], 'Fecha_modificacion': [fecha_modificacion]})
    df_valores = pd.concat([df_valores, df_valores_new])
df_valores = df_valores.sort_values(by='Fecha_modificacion', ascending=True).reset_index(drop=True)
display('df_valores', df_valores)

valores = df_valores['Valor'].tolist()
valores

In [ ]:
#sys.exit()

In [ ]:
# escribe un numero en notacion cientifica con n decimales
def notacion_cientifica(numero, decimales = 2):
    if numero == 0:
        return '0'
    else:
        exponente = int(np.floor(np.log10(abs(numero))))
        base = numero / (10 ** exponente)
        return f'{base:.{decimales}f} x E{exponente}'

In [ ]:
fecha_inicial = '2024-01-01'

In [ ]:
for valor in valores:
    print('Valor', valor)
    # N: Cantidad de "soportes" ocupados
    N = n_sizes[valor]
    
    #if prueba260331:
    #    N = 40
    #sizes = (M, N)
    #print(sizes)

    # 1. Leer el CSV
    df = pd.read_csv(f'{carpeta_data}{valor}.csv')
    df = df.sort_values(by='DateTime').reset_index(drop=True)

    # 2. Convertir la columna DateTime a datetime
    df['DateTime'] = pd.to_datetime(df['DateTime'])

    # 3. Establecer DateTime como índice
    df_plot = df.set_index('DateTime')

    if display_0:
        display('All gráfico', df)
        mpf.plot(df_plot, type = 'candle', volume = False, style = 'charles', figsize = (21, 7), datetime_format='%Y-%m-%d')

    df = df[df['DateTime'] >= fecha_inicial].reset_index(drop=True) # Para no tomar un periodo tan antiguo
    ultimo_cierre = float(df['Close'].iloc[-1])
    ultimo_periodo = df['DateTime'].iloc[-1]
    dt_min = df['DateTime'].min()
    # Pendiente, cuando parámetros en nueva versión estén claros (ver este fragmento abajo en ANEXO 1)
    # df_parametros (una fila, con todos los parámetros actuales)
    # crear_fila_parametros(df_parametros, diccionario_parametros)

    df_plot_inicial = df.set_index('DateTime')

    dt_max = df['DateTime'].max()

    # Normalización de t, como referencia desde el inicio del periodo
    df['t'] = (df['DateTime'] - dt_min).dt.total_seconds() / 3600  # tiempo en horas desde el inicio
    df['t'] = df['t'] / df['t'].max()  # normalizar entre 0 y 1

    conjunto_N_preliminar = set()
    n_randoms_iniciales = 1000
    if f'{valor}_{N}_beta.pkl' in os.listdir('conjuntosN2/'):
        print('Se obtiene resultado ya existente')
        conjunto_N_preliminar = pickle_act(f'conjuntosN2/{valor}_{N}_beta')
        n_randoms_iniciales = 0
    
    #sys.exit(n_randoms_iniciales)

    print(f'INFO: Valor: {valor}, N: {N}, M: {M}, len(conjunto_N_preliminar): {len(conjunto_N_preliminar)}, n_randoms_iniciales: {n_randoms_iniciales}')

    if display_1:
        print(f'Valor: {valor}. Desde {fecha_inicial}')
        mpf.plot(df_plot_inicial, type = 'candle', volume = False, style = 'charles', figsize = (21, 7))

    OCP = 0
    print('obtener_df_extremos')
    df_extremos, conjunto_N = obtener_df_extremos_V2(df, k, n, N, M, OCP, conjunto_N_preliminar, prints_general = True) # OCP es la cantidad de ordenes de compra planificadas (ya activadas en la plataforma) # Entrega conjunto_N y conjunto_M iniciales, dependiendo de los casos (puede que ya exista un conjunto_N precio, y se aprovecha para el algorítmo)
    print('FO_old [guardada comparativa]')
    print(f'INFO: Valor: {valor}, N: {N}, M: {M}, len(conjunto_N): {len(conjunto_N)}, n_randoms_iniciales: {n_randoms_iniciales}')
    #c0 = conjunto_N.copy()
    # Solo para conocer el valor de la FO obtenida con el algorítmo original de reemplazo 1 a 1
    buscar_soportes_optimos_v5(df_extremos, conjunto_N, lambda_ponderador, n_randoms_iniciales = n_randoms_iniciales)  # Se obtiene la mejor combinación de soportes óptimos
    #print(f'INFO: Valor: {valor}, N: {N}, M: {M}, len(conjunto_N): {len(conjunto_N)}, n_randoms_iniciales: {n_randoms_iniciales}')
    #c1 = conjunto_N.copy()
    #print('Nueva FO partiendo de una base random')
    #if valor != "TSLA":
    #    conjunto_N = set() # Small batch (por ahora)
    #c2 = conjunto_N.copy()
    conjunto_N, df_extremos, df_FO = nuevo_optimizador_2(N, df_extremos, conjunto_N, lambda_ponderador, ordenes_activas = [], M = M, max_iters = max_iters)
    #print(f'INFO: Valor: {valor}, N: {N}, M: {M}, len(conjunto_N): {len(conjunto_N)}, n_randoms_iniciales: {n_randoms_iniciales}')
    graficar_df_extremos(df_extremos, graficar = graficar_extremos) # Se grafican los pesos asociados a cada extremo

    # H_n son las distancias entre los soportes en conjunto_N
    L_n = list(conjunto_N)
    L_n.sort()
    H_n = [L_n[i] - L_n[i-1] for i in range(1, len(L_n))]

    #display('df_extremos', df_extremos)
    print('Performance FO')
    graficar_performance_FO(df_FO, graficar = graficar_perf_FO) # Gráfico del comportamiento de FO
    graficar_soportes_all(df, conjunto_N, graficar = graficar_soportes) # Gráfico de soportes sobre el precio

    conjunto_N
    # Se guarda el conjunto actual
    pickle_act((f'conjuntosN2/{valor}_{N}_beta'), conjunto_N, 'save')

  1%|          | 1/130 [01:06<2:22:46, 66.40s/it]

Factor de mejora 0.0005477295948212495 minimo 0.0001
Nuevo FO 83: 5.0173 x E-3 | particion = [5.5531 x E-3, 2.6789 x E-1] | con cambio en i_change = 61, n=81363.0 por caso=81236.53



  0%|          | 0/130 [00:39<?, ?it/s]

Factor de mejora 0.000347069230119334 minimo 0.0001
Nuevo FO 84: 5.0191 x E-3 | particion = [5.5528 x E-3, 2.6688 x E-1] | con cambio en i_change = 107, n=109940.0 por caso=110062.73



  5%|▍         | 6/130 [03:13<1:06:40, 32.26s/it]

...


  2%|▏         | 3/130 [02:21<1:39:41, 47.10s/it]

Factor de mejora 0.0001674281590263082 minimo 0.0001
Nuevo FO 94: 5.0353 x E-3 | particion = [5.5534 x E-3, 2.5907 x E-1] | con cambio en i_change = 49, n=72484.0 por caso=72532.74



  2%|▏         | 3/130 [02:28<1:45:01, 49.62s/it]

Factor de mejora 0.00029803515066813144 minimo 0.0001
Nuevo FO 95: 5.0368 x E-3 | particion = [5.5535 x E-3, 2.5834 x E-1] | con cambio en i_change = 56, n=77876.0 por caso=77778.08



  2%|▏         | 3/130 [02:02<1:31:59, 43.46s/it]

In [ ]:
# Crea una funcion que determine los elementos duplicados en una lista
def elementos_duplicados(lista):
    elementos_vistos = set()
    duplicados = set()

    for elemento in lista:
        if elemento in elementos_vistos:
            duplicados.add(elemento)
        else:
            elementos_vistos.add(elemento)

    return list(duplicados)


lista = [138.79, 144.38790299375646, 145.61962712021366, 148.73649232350033, 157.70500396317195, 160.42304047088763, 160.5772638795086, 168.81525885955728, 175.8866351259884, 177.05851671534, 177.3918479519402, 181.14997459129407, 189.58297294140527, 190.24280666602468, 201.510660836625, 205.07056108076637, 206.63623005710537, 210.05835434505994, 224.36061863329496, 233.84050245209733, 236.65638157163065, 241.45296868283788, 244.13560164980174, 245.8413449571159, 248.35511299563194, 250.26736906818326, 252.08029788400634, 260.4364724625244, 261.904510941141, 262.80163206066567, 284.8382920818453, 286.57533181047245, 287.0492820448648, 287.5314376017867, 291.0637427073987, 291.2776651847865, 300.6628336810013, 321.61860149356517, 329.7698503318053, 334.023640197396, 334.41674399445617, 339.9486953946206, 347.1359367901698, 347.8651445134472, 348.24819733803474, 348.44059472307583, 349.5147644442432, 351.33910729548177, 354.7870818896241, 359.91200401239144, 371.6123723869439, 372.7490820456594, 382.4035367832042, 385.0356050381871, 387.5476148905575, 391.6279297888966, 392.84870154074065, 393.51701416654237, 403.76037506360376, 404.5330409851399, 405.73145279401933, 411.57073352713087, 412.54999734805574, 419.24642670820947, 423.84004969313787, 428.8076889931423, 428.8076889931423, 435.4631278696288, 436.23880775463317, 444.71425000631154, 460.60007965989075, 463.76510129922417, 466.25725838569645, 480.4879587578465, 491.6307961177358]

elementos_duplicados(lista)

In [ ]:
lista = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
for i in range(75):
    if i not in lista:
        print(i)

len(lista)

In [ ]:
#GOOGL: 0.04360930172176972
# nvda: 0.02304030162863239

Cambio de versión respaldo a productiva

In [ ]:
import shutil

In [ ]:
input('Presionar cualquier cosa para continuar')

In [ ]:
for valor in valores:
    print('Valor', valor)
    # N: Cantidad de "soportes" ocupados
    N = n_sizes[valor]
    archivo = f'{valor}_{N}_beta.pkl'
    nuevo_nombre = f'{valor}_{N}.pkl'
    if archivo in os.listdir('conjuntosN2/'):
        try:
            os.remove(os.path.join('conjuntosN2/', nuevo_nombre))
        except:
            None
        # copia archivo a nuevo_nombre (mantener el original)
        os.rename(os.path.join('conjuntosN2/', archivo), os.path.join('conjuntosN2/', nuevo_nombre))
        shutil.copy(os.path.join('conjuntosN2/', nuevo_nombre), os.path.join('conjuntosN2/', archivo))
    else:
        print('No está', archivo)
        print(os.listdir('conjuntosN2/'))

Macroalgorítmo

In [ ]:
# Comenzar con t = T (60 días)

# Para cada t in range(T, len(df)):
#   0. definir el df entre t = 0 y t = t (A0)
#   # 1. Si t está en lista_cambios_soporte (cada d días, o d periodos) (t=T debe estar en la lista [ajuste inicial]) (A1)
#       # calcular soportes óptimos
#           # Este algorítmo tiene OA (Ordenes activas) y OCP (Ordenes de compra planificadas [están esperando que el precio las toque para activarse, ya están programadas como ordenes de compra en la plataforma])
#           # El objetivo es, con M puntos potenciales, llegar a que OCE + OCP = N soportes óptimos [OCE: Ordenes de compra en espera]
#           # 1.1 (Requiere algorítmo) Las ordenes de compra (precio = P) en espera, estarán sobre el precio actual y, se esperará a que el precio llegue a P+delta, para que se transforme en una OCP
    # De esta forma, la idea es ir buscando ordenes de compra y poder activarlas
    
    # 2. Cambio de SL
    #    Para cada OA de precio P
    #       Si p(t) [Precio en t], es mayor a P + delta1 y SL(OA) < P + delta2: (Primera configuración de SL que implique ganancia mínima) -> SL_new(OA) > Precio de compra
    #            SL(OA) = P + delta2 (delta1 > delta2)
    #       Si p(t) es mayor a SL(OA) + delta3: # Trailing stop, mover el stop loss hacia arriba
    #            SL(OA) = SL(OA) + delta4
    
    # 3. Evaluar valor en cuenta
    
    # 4. Evaluar cambios en los parámetros


## Estado de las ordenes de compra:
# OP: Orden potencial (pertenece al conjunto M inicial, donde solo algunas de ellas, se eligen como soportes óptimos)
# OCE: Ordenes de compra en espera (pertenecen al conjunto N de soportes óptimos, que están sobre el precio actual - delta. Cuando el precio actual supera a
#      P + delta, se transforman en OCP [P es el precio de la OCE]
# OCP: Ordenes de compra planificadas (ya están creadas en la plataforma, esperando que el precio las toque para activarse)
# OA: Ordenes abiertas (ya se ejecutaron y están abiertas)
# OC: Ordenes cerradas (se cerraron las ordenes abiertas)
#     Estás últimas se separan en 
#     OCL: Orden cerrada con loss
#     OCW: Orden cerrada con win

In [ ]:
fuente = 'conjuntosN2' # carpeta donde se guardan los conjuntos de soportes y resistencias para cada valor y cada N

In [ ]:
def leer_lista_N(valor, N, fuente = fuente):
    #print(f'Lectura conjuntosN/{valor}_{N}_beta')
    for i in range(10):
        if f'{valor}_{N}_beta.pkl' in os.listdir(f'{fuente}/'):
            lista_N = list(pickle_act(f'{fuente}/{valor}_{N}_beta'))
            lista_N = [round(n, 2) for n in lista_N]
            break
        time.sleep(2) # sleep de 2 segundos

    return lista_N

In [ ]:
valores = ['BTCUSD', 'ETHUSD', 'TSLA', 'GOOGL', 'NVDA', 'AMZN']

In [ ]:
for valor in valores:
    N = n_sizes[valor]
    lista_N = leer_lista_N(valor, N, fuente = fuente)
    lista_N.sort()
    df_lista_N = pd.DataFrame(lista_N, columns = ['Precio'])
    display(valor, N, df_lista_N)